# Prédiction de l'activité anti-HIV par Deep Learning
### Dataset MolNet HIV — DeepChem

**Référence théorique :** Li et al., *Deep learning methods for molecular representation and property prediction*, Drug Discovery Today, 2022

**Modèles implémentés :**
- Baseline : Random Forest + CircularFingerprint (ECFP4)
- GraphConv (GCN spatial)
- AttentiveFP (GCN + mécanisme d'attention)
- MPNN (Message Passing Neural Network)

**Métrique principale :** ROC-AUC (classification binaire déséquilibrée, ratio ~1:28)

---

## 0. Installation et imports

In [ ]:
# Installation (décommenter si nécessaire)
#!pip install deepchem tensorflow==2.15.0 tf-keras==2.15.1 "protobuf>=3.20,<7" dgl dgllife scikit-learn matplotlib seaborn

In [ ]:

import os
import warnings
warnings.filterwarnings('ignore')

# Compat DeepChem avec Keras 3: doit être défini AVANT import tensorflow/deepchem.
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import deepchem as dc
from deepchem.models import GraphConvModel, AttentiveFPModel, MPNNModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    classification_report, confusion_matrix
)

import torch
import random

# Reproductibilité
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'DeepChem version : {dc.__version__}')
print(f'PyTorch version  : {torch.__version__}')
print(f'Device           : {DEVICE}')


## 1. Chargement et exploration du dataset HIV

Le dataset HIV provient du **DTP AIDS Antiviral Screen** du National Cancer Institute.  
Chaque molécule est labellisée **1** (inhibe la réplication HIV) ou **0** (inactive).  
⚠️ Déséquilibre critique : ~3.5% d'actifs seulement → le ROC-AUC est la métrique obligatoire.

In [ ]:
# ── Chargement avec MolGraphConvFeaturizer (pour GCN) ──────────────────────────
print('Chargement du dataset HIV (featurizer graphe)...')

featurizer_graph = dc.feat.MolGraphConvFeaturizer(use_edges=True)

tasks_graph, datasets_graph, transformers_graph = dc.molnet.load_hiv(
    featurizer=featurizer_graph,
    splitter='scaffold',
    reload=False
)

train_graph, valid_graph, test_graph = datasets_graph

print(f'\nTâche               : {tasks_graph}')
print(f'Train               : {len(train_graph)} molécules')
print(f'Validation          : {len(valid_graph)} molécules')
print(f'Test                : {len(test_graph)} molécules')

# Distribution des classes
y_train_all = train_graph.y.flatten()
n_pos = int(y_train_all.sum())
n_neg = len(y_train_all) - n_pos
ratio = n_neg / n_pos

print(f'\n── Distribution train ──────────────────')
print(f'Actifs (label=1)    : {n_pos:>6}  ({100*n_pos/len(y_train_all):.1f}%)')
print(f'Inactifs (label=0)  : {n_neg:>6}  ({100*n_neg/len(y_train_all):.1f}%)')
print(f'Ratio déséquilibre  : 1 : {ratio:.0f}')

In [ ]:
# ── Construction ECFP à partir des splits déjà chargés (évite un 2e load_hiv) ─────
print('Construction des features ECFP depuis les splits graphe existants...')

featurizer_ecfp = dc.feat.CircularFingerprint(size=1024, radius=2)

def _to_ecfp_dataset(ds, split_name):
    smiles = ds.ids  # smiles du split (déjà scaffold-splitté)
    print(f'  Featurization ECFP — {split_name}: {len(smiles)} molécules')
    X = featurizer_ecfp.featurize(smiles)
    return dc.data.NumpyDataset(X=X, y=ds.y, w=ds.w, ids=smiles)

train_ecfp = _to_ecfp_dataset(train_graph, 'train')
valid_ecfp = _to_ecfp_dataset(valid_graph, 'valid')
test_ecfp  = _to_ecfp_dataset(test_graph,  'test')

tasks_ecfp = tasks_graph
transformers_ecfp = transformers_graph

print('OK — datasets ECFP prêts (sans rechargement complet du dataset).')

In [ ]:
# ── Visualisation de la distribution ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Graphique 1 : barres de distribution
splits = ['Train', 'Validation', 'Test']
for split_name, ds in zip(splits, [train_graph, valid_graph, test_graph]):
    y = ds.y.flatten()
    axes[0].bar(
        split_name,
        [int((y == 0).sum()), int((y == 1).sum())],
        label=split_name
    )

colors = ['#378ADD', '#E24B4A']
labels_class = ['Inactifs (0)', 'Actifs (1)']
x = np.arange(len(splits))
w = 0.35

axes[0].cla()
for i, (split_name, ds) in enumerate(zip(splits, [train_graph, valid_graph, test_graph])):
    y = ds.y.flatten()
    axes[0].bar(i - w/2, int((y == 0).sum()), w, color=colors[0], label='Inactifs' if i==0 else '')
    axes[0].bar(i + w/2, int((y == 1).sum()), w, color=colors[1], label='Actifs' if i==0 else '')

axes[0].set_xticks(range(3))
axes[0].set_xticklabels(splits)
axes[0].set_title('Distribution des classes par split', fontweight='bold')
axes[0].set_ylabel('Nombre de molécules')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Graphique 2 : camembert dataset total
y_total = np.concatenate([train_graph.y, valid_graph.y, test_graph.y]).flatten()
n_act = int(y_total.sum())
n_inact = len(y_total) - n_act
axes[1].pie(
    [n_inact, n_act],
    labels=[f'Inactifs\n{n_inact:,} ({100*n_inact/len(y_total):.1f}%)',
            f'Actifs\n{n_act:,} ({100*n_act/len(y_total):.1f}%)'],
    colors=colors,
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
axes[1].set_title('Dataset total — déséquilibre de classes', fontweight='bold')

plt.suptitle('Dataset HIV — MolNet', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('hiv_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Total dataset : {len(y_total):,} molécules — ratio 1:{n_inact//n_act}')

## 2. Baseline — Random Forest + ECFP4

Selon l'article Li 2022 §Hybrid data-based methods, les fingerprints 1D restent une référence solide.  
Ce baseline permet de quantifier le gain réel des GCN en §2D Graph-based methods.

In [ ]:
print('=== MODÈLE 1 : Random Forest baseline (ECFP4) ===')
print()

# Extraction des features et labels
X_train = train_ecfp.X
y_train = train_ecfp.y.flatten().astype(int)
X_valid = valid_ecfp.X
y_valid = valid_ecfp.y.flatten().astype(int)
X_test  = test_ecfp.X
y_test  = test_ecfp.y.flatten().astype(int)

print(f'Shape X_train : {X_train.shape}')
print(f'Shape X_test  : {X_test.shape}')

# Poids de classe pour compenser le déséquilibre
rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    class_weight='balanced',  # compense le ratio 1:28
    n_jobs=-1,
    random_state=SEED
)

print('\nEntraînement Random Forest...')
rf_model.fit(X_train, y_train)

# Prédictions
y_pred_proba_valid_rf = rf_model.predict_proba(X_valid)[:, 1]
y_pred_proba_test_rf  = rf_model.predict_proba(X_test)[:, 1]

roc_valid_rf = roc_auc_score(y_valid, y_pred_proba_valid_rf)
roc_test_rf  = roc_auc_score(y_test,  y_pred_proba_test_rf)
prc_test_rf  = average_precision_score(y_test, y_pred_proba_test_rf)

print(f'\nRésultats Random Forest :')
print(f'  ROC-AUC validation : {roc_valid_rf:.4f}')
print(f'  ROC-AUC test       : {roc_test_rf:.4f}')
print(f'  AUPRC test         : {prc_test_rf:.4f}')

## 3. GraphConv — GCN spatial

**Référence article :** §Spatial GCN models — GraphConv implémente le message passing spatial où chaque atome agrège les features de ses voisins sur plusieurs couches.

In [ ]:
print('=== MODÈLE 2 : GraphConvModel (GCN spatial) ===')
print()

# GraphConvModel DeepChem attend des objets ConvMol (pas GraphData).
convmol_featurizer = dc.feat.ConvMolFeaturizer()

def _to_convmol_dataset(ds, split_name):
    smiles = ds.ids
    print(f'  Conversion ConvMol — {split_name}: {len(smiles)} molécules')
    X = convmol_featurizer.featurize(smiles)
    return dc.data.NumpyDataset(X=X, y=ds.y, w=ds.w, ids=smiles)

train_graphconv = _to_convmol_dataset(train_graph, 'train')
valid_graphconv = _to_convmol_dataset(valid_graph, 'valid')
test_graphconv  = _to_convmol_dataset(test_graph,  'test')

# Patch TensorFlow BatchNormalization pour compatibilité DeepChem (arg 'fused').
# Le patch est idempotent et évite la récursion si la cellule est relancée.
from tensorflow.keras.layers import BatchNormalization

if not hasattr(BatchNormalization, '_original_init_unpatched'):
    BatchNormalization._original_init_unpatched = BatchNormalization.__init__

if not getattr(BatchNormalization.__init__, '_deepchem_fused_patch', False):
    _bn_init_original = BatchNormalization._original_init_unpatched

    def _bn_init_without_fused(self, *args, **kwargs):
        kwargs.pop('fused', None)
        return _bn_init_original(self, *args, **kwargs)

    _bn_init_without_fused._deepchem_fused_patch = True
    BatchNormalization.__init__ = _bn_init_without_fused

# Calcul du class_weight pour compenser le déséquilibre
pos_weight = n_neg / n_pos  # ~28
print(f'class_weight positif : {pos_weight:.1f}x')

gc_model = GraphConvModel(
    n_tasks=1,
    mode='classification',
    graph_conv_layers=[128, 128],   # 2 couches de convolution
    dense_layer_size=256,
    dropout=0.2,
    learning_rate=1e-3,
    batch_size=128,
    class_imbalance_ratio=pos_weight,
    device=DEVICE
)

print('Entraînement GraphConv (30 epochs)...')
best_roc_gc = 0
gc_train_history = []
gc_valid_history = []

for epoch in range(1, 31):
    gc_model.fit(train_graphconv, nb_epoch=1, deterministic=True)

    train_scores = gc_model.evaluate(train_graphconv, [dc.metrics.Metric(dc.metrics.roc_auc_score)])
    valid_scores = gc_model.evaluate(valid_graphconv, [dc.metrics.Metric(dc.metrics.roc_auc_score)])

    roc_tr = train_scores['roc_auc_score']
    roc_va = valid_scores['roc_auc_score']
    gc_train_history.append(roc_tr)
    gc_valid_history.append(roc_va)

    if roc_va > best_roc_gc:
        best_roc_gc = roc_va
        gc_model.save_checkpoint(model_dir='./checkpoints_gc')

    if epoch % 5 == 0:
        print(f'  Epoch {epoch:2d}/30 — train ROC: {roc_tr:.4f} | valid ROC: {roc_va:.4f} {"✓ best" if roc_va == best_roc_gc else ""}')

# Chargement du meilleur checkpoint
gc_model.restore(model_dir='./checkpoints_gc')

# Évaluation finale
y_pred_gc = gc_model.predict(test_graphconv)
y_pred_proba_test_gc = y_pred_gc[:, 0, 1]  # proba classe 1

roc_test_gc = roc_auc_score(test_graphconv.y.flatten(), y_pred_proba_test_gc)
prc_test_gc = average_precision_score(test_graphconv.y.flatten(), y_pred_proba_test_gc)

print(f'\nRésultats GraphConv (meilleur checkpoint) :')
print(f'  ROC-AUC test : {roc_test_gc:.4f}')
print(f'  AUPRC test   : {prc_test_gc:.4f}')

## 4. AttentiveFP — GCN + mécanisme d'attention

**Référence article :** §Spatial GCN models — AttentiveFP "automatically learns nonlocal intramolecular interactions and captures hidden edges through an attention mechanism" (Li et al. 2022).  
C'est le modèle le plus interprétable : les poids d'attention indiquent quels atomes contribuent à la prédiction.

In [ ]:
print('=== MODÈLE 3 : AttentiveFPModel (GCN + attention) ===')
print()

AFP_AVAILABLE = True
try:
    import dgl  # noqa: F401
    import dgllife  # noqa: F401
except ImportError:
    AFP_AVAILABLE = False
    print('DGL/DGLLife non installés: AttentiveFP est ignoré.')
    print('Installe-les avec: pip install dgl dgllife')

if AFP_AVAILABLE:
    afp_model = AttentiveFPModel(
        n_tasks=1,
        mode='classification',
        num_layers=3,          # profondeur du graphe de molécule
        num_timesteps=2,       # itérations de readout par attention
        graph_feat_size=200,   # taille du vecteur de représentation
        dropout=0.2,
        learning_rate=1e-3,
        batch_size=128,
        device=DEVICE
    )

    print('Entraînement AttentiveFP (30 epochs)...')
    best_roc_afp = 0
    afp_train_history = []
    afp_valid_history = []

    for epoch in range(1, 31):
        afp_model.fit(train_graph, nb_epoch=1, deterministic=True)

        train_scores = afp_model.evaluate(train_graph, [dc.metrics.Metric(dc.metrics.roc_auc_score)])
        valid_scores = afp_model.evaluate(valid_graph, [dc.metrics.Metric(dc.metrics.roc_auc_score)])

        roc_tr = train_scores['roc_auc_score']
        roc_va = valid_scores['roc_auc_score']
        afp_train_history.append(roc_tr)
        afp_valid_history.append(roc_va)

        if roc_va > best_roc_afp:
            best_roc_afp = roc_va
            afp_model.save_checkpoint(model_dir='./checkpoints_afp')

        if epoch % 5 == 0:
            print(f'  Epoch {epoch:2d}/30 — train ROC: {roc_tr:.4f} | valid ROC: {roc_va:.4f} {"✓ best" if roc_va == best_roc_afp else ""}')

    afp_model.restore(model_dir='./checkpoints_afp')

    y_pred_afp = afp_model.predict(test_graph)
    y_pred_proba_test_afp = y_pred_afp[:, 1]

    roc_test_afp = roc_auc_score(test_graph.y.flatten(), y_pred_proba_test_afp)
    prc_test_afp = average_precision_score(test_graph.y.flatten(), y_pred_proba_test_afp)

    print(f'\nRésultats AttentiveFP (meilleur checkpoint) :')
    print(f'  ROC-AUC test : {roc_test_afp:.4f}')
    print(f'  AUPRC test   : {prc_test_afp:.4f}')
else:
    afp_train_history = []
    afp_valid_history = []
    y_pred_proba_test_afp = np.zeros(len(test_graph.y.flatten()), dtype=float)
    roc_test_afp = float('nan')
    prc_test_afp = float('nan')
    print('AttentiveFP non exécuté (dépendance manquante).')

In [ ]:
# Vérification des variables nécessaires
if 'afp_model' not in globals():
    raise RuntimeError(
        "afp_model n'est pas disponible. Relance d'abord la cellule d'entraînement AttentiveFP ou restaure son checkpoint."
    )

if 'test_graph' not in globals():
    raise RuntimeError("test_graph n'est pas disponible.")

# Prédictions
y_pred_afp = afp_model.predict(test_graph)

# Gestion des différentes formes de sortie
if y_pred_afp.ndim == 3:
    y_pred_proba_test_afp = y_pred_afp[:, 0, 1]
elif y_pred_afp.ndim == 2 and y_pred_afp.shape[1] >= 2:
    y_pred_proba_test_afp = y_pred_afp[:, 1]
else:
    y_pred_proba_test_afp = y_pred_afp.reshape(-1)

# Calcul des métriques
from sklearn.metrics import roc_auc_score, average_precision_score

y_true = test_graph.y.flatten()

roc_test_afp = roc_auc_score(y_true, y_pred_proba_test_afp)
prc_test_afp = average_precision_score(y_true, y_pred_proba_test_afp)

# Affichage des résultats
print(f"ROC-AUC test : {roc_test_afp:.4f}")
print(f"AUPRC test : {prc_test_afp:.4f}")

## 6. Comparaison des modèles — Courbes ROC et PR

Visualisation complète selon les préconisations de l'article pour les datasets déséquilibrés :
- **Courbe ROC** : sensibilité vs spécificité
- **Courbe Précision-Rappel** : plus informative sur les classes minoritaires

In [ ]:
# Imports nécessaires
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve

# ── Regroupement des résultats (sans MPNN) ────────────────────────────────────
results = {
    'Random Forest': {
        'proba': y_pred_proba_test_rf,
        'roc':   roc_test_rf,
        'prc':   prc_test_rf,
        'color': '#888780',
        'ls':    '--'
    },
    'GraphConv': {
        'proba': y_pred_proba_test_gc,
        'roc':   roc_test_gc,
        'prc':   prc_test_gc,
        'color': '#378ADD',
        'ls':    '-'
    },
    'AttentiveFP': {
        'proba': y_pred_proba_test_afp,
        'roc':   roc_test_afp,
        'prc':   prc_test_afp,
        'color': '#1D9E75',
        'ls':    '-'
    }
}

# Vérité terrain
y_true = test_graph.y.flatten()

# Création des subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Courbe ROC ────────────────────────────────────────────────────────────────
ax = axes[0]
ax.plot([0, 1], [0, 1], 'k:', alpha=0.4, label='Aléatoire (AUC=0.50)')

for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_true, res['proba'])
    ax.plot(
        fpr, tpr,
        color=res['color'], linestyle=res['ls'], linewidth=2,
        label=f"{name} (AUC={res['roc']:.4f})"
    )

ax.set_xlabel('Taux de faux positifs', fontsize=12)
ax.set_ylabel('Taux de vrais positifs', fontsize=12)
ax.set_title('Courbe ROC — Dataset HIV test', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

# ── Courbe Précision-Rappel ───────────────────────────────────────────────────
ax = axes[1]
baseline_pr = y_true.sum() / len(y_true)

ax.axhline(
    y=baseline_pr,
    color='k',
    linestyle=':',
    alpha=0.4,
    label=f'Baseline aléatoire (P={baseline_pr:.3f})'
)

for name, res in results.items():
    prec, rec, _ = precision_recall_curve(y_true, res['proba'])
    ax.plot(
        rec, prec,
        color=res['color'], linestyle=res['ls'], linewidth=2,
        label=f"{name} (AUPRC={res['prc']:.4f})"
    )

ax.set_xlabel('Rappel', fontsize=12)
ax.set_ylabel('Précision', fontsize=12)
ax.set_title('Courbe Précision-Rappel — Dataset HIV test', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

# Titre global
plt.suptitle(
    'Comparaison des modèles — HIV MolNet (scaffold split)',
    fontsize=14,
    fontweight='bold',
    y=1.02
)

plt.tight_layout()
plt.savefig('hiv_curves_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Imports nécessaires
import matplotlib.pyplot as plt
import numpy as np

# ── Courbes d'apprentissage des modèles GCN (sans MPNN) ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

histories = {
    'GraphConv':   (gc_train_history,   gc_valid_history,   '#378ADD'),
    'AttentiveFP': (afp_train_history,  afp_valid_history,  '#1D9E75'),
}

for ax, (name, (tr, va, color)) in zip(axes, histories.items()):
    if len(tr) == 0 or len(va) == 0:
        ax.text(0.5, 0.5, f'{name}\nnon exécuté', ha='center', va='center', fontsize=11)
        ax.set_title(name, fontweight='bold', fontsize=11)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('ROC-AUC')
        ax.set_xlim([0, 1])
        ax.set_ylim([0, 1])
        ax.grid(alpha=0.3)
        continue

    epochs = range(1, len(tr) + 1)

    ax.plot(epochs, tr, color=color, alpha=0.5, linestyle='--', label='Train')
    ax.plot(epochs, va, color=color, linewidth=2, label='Validation')

    best_ep = np.argmax(va) + 1
    ax.axvline(x=best_ep, color='gray', linestyle=':', alpha=0.7)
    ax.scatter([best_ep], [max(va)], color=color, s=80, zorder=5)

    ax.set_title(
        f'{name}\nBest valid: {max(va):.4f} (ep.{best_ep})',
        fontweight='bold',
        fontsize=11
    )
    ax.set_xlabel('Epoch')
    ax.set_ylabel('ROC-AUC')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_ylim([0.4, 1.0])

plt.suptitle("Courbes d'apprentissage (ROC-AUC)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('hiv_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Tableau récapitulatif des performances

In [ ]:
# Imports
import pandas as pd

# ── Tableau comparatif (sans MPNN) ────────────────────────────────────────────
summary = []

for name, res in results.items():
    y_pred_bin = (res['proba'] >= 0.5).astype(int)

    summary.append({
        'Modèle': name,
        'Représentation': 'ECFP4 (1D)' if name == 'Random Forest' else 'Graphe moléculaire (2D)',
        'ROC-AUC': round(res['roc'], 4),
        'AUPRC': round(res['prc'], 4),
        'Référence article': (
            'Li 2022 §Hybrid' if name == 'Random Forest'
            else 'Li 2022 §Spatial GCN'
        )
    })

# DataFrame
df_summary = pd.DataFrame(summary).sort_values('ROC-AUC', ascending=False)
df_summary = df_summary.reset_index(drop=True)
df_summary.index = df_summary.index + 1

# Affichage
print('=' * 75)
print('RÉSULTATS FINAUX — HIV MolNet (scaffold split, test set)')
print('=' * 75)
print(df_summary.to_string(index=True))
print('=' * 75)

# Meilleur modèle
best_model = df_summary.iloc[0]['Modèle']
best_roc = df_summary.iloc[0]['ROC-AUC']

# Baseline Random Forest
rf_roc = df_summary[df_summary['Modèle'] == 'Random Forest']['ROC-AUC'].values[0]
gain = best_roc - rf_roc

print(f'\nMeilleur modèle : {best_model} (ROC-AUC = {best_roc:.4f})')
print(f'Gain vs baseline RF : +{gain:.4f} ({gain*100:.1f} points)')

In [ ]:
# Imports
import matplotlib.pyplot as plt

# ── Visualisation barres comparatives (sans MPNN) ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

model_names = [r['Modèle'] for r in summary]
roc_vals    = [r['ROC-AUC'] for r in summary]
prc_vals    = [r['AUPRC'] for r in summary]

# Couleurs pour 3 modèles uniquement
bar_colors  = ['#888780', '#378ADD', '#1D9E75']

for ax, vals, title, baseline in zip(
    axes,
    [roc_vals, prc_vals],
    ['ROC-AUC (test)', 'AUPRC (test)'],
    [0.5, baseline_pr]
):
    bars = ax.bar(
        model_names,
        vals,
        color=bar_colors,
        edgecolor='white',
        linewidth=1.5
    )

    ax.axhline(
        y=baseline,
        color='gray',
        linestyle=':',
        alpha=0.6,
        label='Aléatoire'
    )

    # Affichage des valeurs au-dessus des barres
    for bar, val in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f'{val:.4f}',
            ha='center',
            va='bottom',
            fontsize=10,
            fontweight='bold'
        )

    ax.set_ylim([0, min(1.0, max(vals) + 0.12)])
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel(title.split(' ')[0])
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=9)
    ax.tick_params(axis='x', rotation=10)

plt.suptitle('Comparaison des performances — HIV MolNet', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('hiv_performance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Interprétabilité — Importances atomiques (AttentiveFP)

**Référence article :** §Interpretability — AttentiveFP permet de visualiser les poids d'attention par atome,
révélant quels groupes fonctionnels contribuent à la prédiction d'activité anti-HIV.
Cela correspond aux *feature attribution methods* décrites dans Li 2022.

In [ ]:
try:
    from rdkit import Chem
    from rdkit.Chem import Draw
    from IPython.display import display
    import numpy as np
    RDKIT_OK = True
except ImportError:
    RDKIT_OK = False
    print('RDKit non disponible — section interprétabilité ignorée.')

if RDKIT_OK:
    # Vérifications de sécurité
    if 'test_graph' not in globals() or 'y_pred_proba_test_afp' not in globals():
        raise RuntimeError("Variables nécessaires manquantes (test_graph ou prédictions AFP).")

    # Sélection des molécules actives
    y_test_flat = test_graph.y.flatten()
    active_idx = np.where(y_test_flat == 1)[0]

    smiles_test = test_graph.ids  # SMILES

    print(f'Molécules actives dans le test set : {len(active_idx)}')
    print('\nTop 5 molécules actives les mieux prédites par AttentiveFP :')

    # Probabilités des molécules actives
    active_probas = [(idx, y_pred_proba_test_afp[idx]) for idx in active_idx]
    active_probas.sort(key=lambda x: x[1], reverse=True)

    # Affichage Top 5
    for rank, (idx, prob) in enumerate(active_probas[:5], 1):
        smi = smiles_test[idx]
        print(f'  #{rank}  proba={prob:.4f}  SMILES={smi[:60]}...')

    # Visualisation des Top molécules
    from PIL import Image as PILImage
import io

if len(top_mols) > 0:
    img = Draw.MolsToGridImage(
        top_mols,
        molsPerRow=3,
        subImgSize=(350, 280),
        legends=top_legends
    )

    # Conversion sécurisée + sauvegarde
    if hasattr(img, "save"):
        img.save('hiv_top_actifs.png')
    else:
        # Cas où img est un objet IPython (bytes)
        img = PILImage.open(io.BytesIO(img.data))
        img.save('hiv_top_actifs.png')

    display(img)
    print('\nMolécules actives sauvegardées dans hiv_top_actifs.png')

In [ ]:
if RDKIT_OK:
    import numpy as np

    # Vérification des variables nécessaires
    if 'y_test_flat' not in globals():
        y_test_flat = test_graph.y.flatten()

    if 'y_pred_proba_test_afp' not in globals():
        raise RuntimeError("Prédictions AttentiveFP manquantes.")

    # ── Analyse de la matrice de confusion ────────────────────────────────────
    threshold = 0.5

    fn_idx = np.where((y_test_flat == 1) & (y_pred_proba_test_afp < threshold))[0]
    tp_idx = np.where((y_test_flat == 1) & (y_pred_proba_test_afp >= threshold))[0]
    fp_idx = np.where((y_test_flat == 0) & (y_pred_proba_test_afp >= threshold))[0]
    tn_idx = np.where((y_test_flat == 0) & (y_pred_proba_test_afp < threshold))[0]

    print(f'=== Matrice de confusion AttentiveFP (seuil={threshold}) ===')
    print(f'  Vrais Positifs  (TP) : {len(tp_idx):>5}  (actifs correctement détectés)')
    print(f'  Faux Négatifs   (FN) : {len(fn_idx):>5}  (actifs manqués — critique)')
    print(f'  Faux Positifs   (FP) : {len(fp_idx):>5}  (inactifs classés actifs)')
    print(f'  Vrais Négatifs  (TN) : {len(tn_idx):>5}  (inactifs correctement rejetés)')
    print()

    # Métriques
    recall = len(tp_idx) / (len(tp_idx) + len(fn_idx)) if (len(tp_idx) + len(fn_idx)) > 0 else 0
    prec   = len(tp_idx) / (len(tp_idx) + len(fp_idx)) if (len(tp_idx) + len(fp_idx)) > 0 else 0

    print(f'  Sensibilité (Recall) : {recall:.3f}')
    print(f'  Précision            : {prec:.3f}')
    print()

    # ── Analyse métier ────────────────────────────────────────────────────────
    print('Interprétation :')
    print('- En drug discovery, les Faux Négatifs (FN) sont les plus critiques.')
    print('- Un FN = molécule potentiellement active rejetée ❗')
    print('- Solution : diminuer le seuil (ex: 0.3) pour augmenter le recall.')

    # ── Bonus : test avec seuil plus bas ──────────────────────────────────────
    threshold_low = 0.3

    fn_low = np.where((y_test_flat == 1) & (y_pred_proba_test_afp < threshold_low))[0]
    tp_low = np.where((y_test_flat == 1) & (y_pred_proba_test_afp >= threshold_low))[0]

    recall_low = len(tp_low) / (len(tp_low) + len(fn_low)) if (len(tp_low) + len(fn_low)) > 0 else 0

    print()
    print(f'>>> Avec seuil = {threshold_low} → Recall = {recall_low:.3f}')

## 9. Analyse de l'impact du seuil de décision

Avec un dataset déséquilibré, le seuil par défaut de 0.5 n'est pas optimal.  
En drug discovery, on privilégie souvent le **recall** (ne pas manquer d'actifs).

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Vérifications
if 'y_pred_proba_test_afp' not in globals():
    raise RuntimeError("Prédictions AttentiveFP manquantes.")

if 'y_test_flat' not in globals():
    y_test_flat = test_graph.y.flatten()

# ── Analyse threshold pour AttentiveFP ───────────────────────────────────────
thresholds = np.arange(0.1, 0.9, 0.05)
threshold_results = []

for t in thresholds:
    y_bin = (y_pred_proba_test_afp >= t).astype(int)

    tp = int(((y_test_flat == 1) & (y_bin == 1)).sum())
    fn = int(((y_test_flat == 1) & (y_bin == 0)).sum())
    fp = int(((y_test_flat == 0) & (y_bin == 1)).sum())
    tn = int(((y_test_flat == 0) & (y_bin == 0)).sum())

    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    fnr  = fn / (tp + fn) if (tp + fn) > 0 else 0  # IMPORTANT en pharma

    threshold_results.append({
        'threshold': t,
        'recall': rec,
        'precision': prec,
        'f1': f1,
        'fnr': fnr
    })

df_thresh = pd.DataFrame(threshold_results)

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(df_thresh['threshold'], df_thresh['recall'],    lw=2, label='Recall (sensibilité)')
ax.plot(df_thresh['threshold'], df_thresh['precision'], lw=2, label='Précision')
ax.plot(df_thresh['threshold'], df_thresh['f1'],        lw=2, label='F1-score')

# Meilleur seuil F1
best_f1_row = df_thresh.loc[df_thresh['f1'].idxmax()]

ax.axvline(x=best_f1_row['threshold'], linestyle=':', alpha=0.7)
ax.scatter([best_f1_row['threshold']], [best_f1_row['f1']], s=100, zorder=5)

ax.text(
    best_f1_row['threshold'] + 0.02,
    best_f1_row['f1'],
    f"F1 max={best_f1_row['f1']:.3f}\n(t={best_f1_row['threshold']:.2f})",
    fontsize=9
)

ax.set_xlabel('Seuil de décision', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title(
    'AttentiveFP — Impact du seuil sur Recall / Précision / F1',
    fontsize=12,
    fontweight='bold'
)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_xlim([0.1, 0.85])

plt.tight_layout()
plt.savefig('hiv_threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Résultats ────────────────────────────────────────────────────────────────
print("=== Seuil optimal (max F1) ===")
print(f"Seuil      : {best_f1_row['threshold']:.2f}")
print(f"F1         : {best_f1_row['f1']:.4f}")
print(f"Recall     : {best_f1_row['recall']:.4f}")
print(f"Précision  : {best_f1_row['precision']:.4f}")
print(f"FN rate    : {best_f1_row['fnr']:.4f}")

# ── Bonus métier (TRÈS IMPORTANT) ────────────────────────────────────────────
best_recall_row = df_thresh.loc[df_thresh['recall'].idxmax()]

print("\n=== Seuil orienté DRUG DISCOVERY (max recall) ===")
print(f"Seuil      : {best_recall_row['threshold']:.2f}")
print(f"Recall     : {best_recall_row['recall']:.4f}")
print(f"Précision  : {best_recall_row['precision']:.4f}")
print(f"FN rate    : {best_recall_row['fnr']:.4f}")

## 10. Synthèse et conclusion

Récapitulatif des choix de conception en lien avec l'article Li et al. 2022

In [ ]:
print('=' * 70)
print('SYNTHÈSE DU PROJET — HIV Deep Learning (Li et al. 2022)')
print('=' * 70)

# ── Représentations ──────────────────────────────────────────────────────────
print('\n── Représentations utilisées ──────────────────────────────────────')
print('  1D : ECFP4 (CircularFingerprint r=2, 1024 bits) → Random Forest')
print('       Réf. article : §Fingerprints, §Hybrid data-based methods')
print()
print('  2D : MolGraphConvFeaturizer (30 node feat. + 11 edge feat.) → GCN')
print('       Réf. article : §Spatial GCN models, §Graph-based methods')

# ── Modèles ──────────────────────────────────────────────────────────────────
print('\n── Modèles entraînés ──────────────────────────────────────────────')
for name, res in results.items():
    print(f'  {name:<15} ROC-AUC={res["roc"]:.4f}  AUPRC={res["prc"]:.4f}')

# ── Meilleur modèle ──────────────────────────────────────────────────────────
best_model = max(results.items(), key=lambda x: x[1]['roc'])
print('\n── Meilleur modèle ────────────────────────────────────────────────')
print(f'  {best_model[0]} avec ROC-AUC = {best_model[1]["roc"]:.4f}')

# ── Points clés ──────────────────────────────────────────────────────────────
print('\n── Points clés ─────────────────────────────────────────────────────')
print('  - Scaffold split : évaluation réaliste sur nouvelles structures chimiques')
print('  - Déséquilibre (≈1:28) traité avec class_weight="balanced"')
print('  - Early stopping via validation (checkpoint meilleur epoch)')
print('  - ROC-AUC + AUPRC : métriques adaptées aux données déséquilibrées')
print('  - Ajustement du seuil : compromis recall / précision selon objectif métier')

# ── Insight métier ───────────────────────────────────────────────────────────
print('\n── Insight métier (drug discovery) ─────────────────────────────────')
print('  - Minimiser les Faux Négatifs (FN) est prioritaire')
print('  - Un FN = molécule active potentiellement perdue ❗')
print('  - Seuil plus bas (≈0.3) → meilleur recall → meilleur screening')

# ── Améliorations (article) ─────────────────────────────────────────────────
print('\n── Pistes d\'amélioration (Li et al. 2022) ─────────────────────────')
print('  1. Transfer learning : pré-entraîner sur ChEMBL/ZINC puis fine-tuner')
print('     Réf. : §Transfer learning, multi-task learning')
print('  2. Self-Supervised Learning : MolCLR, GROVER')
print('     Réf. : §Graph-based self-supervised learning methods')
print('  3. Ensemble learning : combiner GraphConv + AttentiveFP')
print('     Réf. : §Ensemble learning')
print('  4. Interprétabilité : visualisation des poids d\'attention')
print('     Réf. : §Interpretability of the DL model')

print()
print('=' * 70)